# Jour 1 · Ingérer, découper et persister des documents


## De nouveaux documents à la base vectorielle

L'**ingestion** désigne les opérations qui préparent les documents pour la recherche. Elle n'est pas exécutée à chaque question.

```text
fichiers → loaders → Documents → splitter → chunks → embeddings → Chroma
```


### Une étape, un rôle

- le **loader** lit un fichier et crée des objets `Document` ;
- le **splitter** découpe leur texte en passages plus petits ;
- le modèle d'**embeddings** transforme chaque passage en vecteur ;
- Chroma stocke le passage, son vecteur et ses métadonnées.

Garder ces rôles séparés aide à repérer précisément l'étape qui pose problème.


![Découpage d'un guide en passages recherchables](../assets/jour_01/03_document_vers_chunks.png)

*Chaque chunk conserve les métadonnées de son document d'origine.*


In [ ]:
from pathlib import Path


def find_course_root() -> Path:
    """Retrouve la racine, même si Jupyter démarre dans un sous-dossier."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "documents").is_dir() and (candidate / "jour_01").is_dir():
            return candidate
    raise FileNotFoundError("Racine du cours introuvable.")


ROOT = find_course_root()
DOCUMENTS_DIR = ROOT / "documents"
OUTPUTS_DIR = ROOT / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

print("Racine du cours :", ROOT)


In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings

CHAT_MODEL = "qwen2.5:3b"
EMBEDDING_MODEL = "bge-m3"

embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
llm = ChatOllama(model=CHAT_MODEL, temperature=0)

print("Modèle de génération :", CHAT_MODEL)
print("Modèle d'embeddings  :", EMBEDDING_MODEL)


## Étape 1 — charger les fichiers

Un **loader** lit un format et renvoie des objets `Document`.

- `TextLoader` lit ici les fichiers Markdown comme du texte.
- `PyPDFLoader` crée généralement un `Document` par page.

Un loader extrait du contenu ; il ne garantit pas qu'un PDF scanné ou une mise en page complexe sera parfaitement compris.


In [ ]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader

documents_bruts = []
for path in sorted(DOCUMENTS_DIR.glob("*.md")):
    if path.name == "README.md":
        continue
    loaded = TextLoader(str(path), encoding="utf-8").load()
    for document in loaded:
        document.metadata["source"] = path.name
    documents_bruts.extend(loaded)

pdf_path = DOCUMENTS_DIR / "guide_escalade_p1.pdf"
for document in PyPDFLoader(str(pdf_path)).load():
    document.metadata["source"] = pdf_path.name
    documents_bruts.append(document)

print("Documents chargés :", len(documents_bruts))
for document in documents_bruts:
    print(document.metadata)


## Étape 2 — pourquoi découper ?

Un guide complet peut parler de connexion, MFA, dépannage et escalade. Un seul embedding mélangerait ces sujets.

- trop petit : l'idée perd son contexte ;
- trop grand : plusieurs sujets se mélangent ;
- trop de chevauchement : la base contient beaucoup de texte répété.


## `RecursiveCharacterTextSplitter`

Le splitter essaie successivement les séparateurs fournis : titres Markdown, paragraphes, lignes, espaces puis caractères.

`chunk_size` est une taille cible maximale mesurée ici en caractères. `chunk_overlap` est une cible ; le résultat exact dépend des séparateurs disponibles.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=450,
    chunk_overlap=80,
    separators=["\n## ", "\n\n", "\n", " ", ""],
)
chunks = splitter.split_documents(documents_bruts)

print("Documents avant découpe :", len(documents_bruts))
print("Chunks après découpe     :", len(chunks))
print("Tailles des cinq premiers:", [len(chunk.page_content) for chunk in chunks[:5]])


In [ ]:
for index, chunk in enumerate(chunks[:4]):
    print(f"\n--- CHUNK {index} · {chunk.metadata} ---")
    print(chunk.page_content)


![Deux chunks partageant une zone de texte](../assets/jour_01/03_overlap.png)

*Le chevauchement protège une idée coupée à la frontière, au prix d'une répétition contrôlée.*


### À vous de jouer — rendre une frontière visible

Créez un petit texte contenant plusieurs paragraphes, puis affichez les morceaux produits.


In [ ]:
# À vous de jouer — ajoutez un texte puis observez sa découpe.
texte_exercice = ""
if texte_exercice:
    for index, morceau in enumerate(splitter.split_text(texte_exercice)):
        print(index, repr(morceau))
else:
    print("Ajoutez un texte pour lancer l'exercice.")


## Étape 3 — créer des identifiants stables

Une base persistante survit au redémarrage. Si chaque relance invente de nouveaux IDs, les mêmes chunks sont ajoutés plusieurs fois.

Nous calculons un hash avec la source, la page et le contenu. Même entrée signifie même ID.


### Un hash ressemble à une empreinte

Un **hash** transforme un contenu en une courte empreinte. Le même contenu produit la même empreinte ; une modification du contenu en produit généralement une autre.

Ici, ce hash sert uniquement d'identifiant reproductible. Ce n'est ni un chiffrement ni une protection du document. Son intérêt est d'éviter qu'une nouvelle ingestion ajoute une seconde copie du même chunk.


![Identifiants aléatoires et identifiant stable](../assets/jour_01/03_ids_stables.png)

*L'idempotence signifie qu'une même ingestion peut être rejouée sans changer le résultat final.*


In [ ]:
import hashlib


def stable_chunk_id(chunk):
    source = chunk.metadata.get("source", "")
    page = chunk.metadata.get("page", "")
    raw = f"{source}|{page}|{chunk.page_content}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


chunk_ids = [stable_chunk_id(chunk) for chunk in chunks]
print("ID 1 :", chunk_ids[0])
print("Même calcul :", stable_chunk_id(chunks[0]))
print("IDs uniques :", len(set(chunk_ids)), "/", len(chunk_ids))


## Étape 4 — persister dans Chroma

Le dossier `outputs/chroma_helpdesk` est ignoré par Git. L'appel avec les mêmes IDs met à jour les mêmes entrées au lieu d'en créer de nouvelles.


In [ ]:
from langchain_chroma import Chroma

database_dir = OUTPUTS_DIR / "chroma_helpdesk"
vectorstore = Chroma(
    collection_name="helpdesk_documents",
    embedding_function=embeddings,
    persist_directory=str(database_dir),
)

avant = len(vectorstore.get()["ids"])
vectorstore.add_documents(chunks, ids=chunk_ids)
apres = len(vectorstore.get()["ids"])

print("Entrées avant :", avant)
print("Entrées après :", apres)
print("Relancez la cellule : le total doit rester stable.")


## Étape 5 — interroger la base persistée

Le programme peut redémarrer et recréer un objet `Chroma` pointant vers le même dossier, sans recalculer tous les embeddings.


In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
question = "Que faut-il joindre au ticket si le VPN reste inaccessible ?"
documents_retrouves = retriever.invoke(question)

for document in documents_retrouves:
    print("\nSOURCE :", document.metadata)
    print(document.page_content)


## À retenir

- Un loader transforme un fichier en objets `Document`.
- Un splitter transforme ces documents en chunks.
- La taille doit être testée sur le contenu réel.
- Les métadonnées doivent survivre au découpage.
- Des IDs stables rendent l'ingestion idempotente.
- Les bases générées restent dans `outputs/`, pas dans Git.
